In [12]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset, Subset


def create_dataset(data):
    labels = data[..., 0]
    features = data[..., 1:]
    features = features.permute(0, 3, 1, 2).contiguous()
    dataset = TensorDataset(features, labels)
    return dataset


def train_val_test_split(dataset, val_indices, test_indices):
    all_indices = set(range(len(dataset)))
    train_indices = list(all_indices - set(val_indices) - set(test_indices))
    train_indices.sort()
    train_dataset = Subset(dataset, train_indices)
    val_dataset = Subset(dataset, val_indices)
    test_dataset = Subset(dataset, test_indices)
    return train_dataset, val_dataset, test_dataset


data = torch.load("data_to_train/tensors.pt")
dataset = create_dataset(data)
val_indices = [20, 33, 52, 71, 90, 110, 132, 155, 177, 197, 219, 237, 250, 257, 22, 39, 58, 77, 96, 116, 138, 161, 183, 203, 225, 243]
test_indices = [21, 34, 53, 72, 91, 111, 133, 154, 178, 198, 220, 238, 251, 258, 23, 40, 59, 78, 97, 117, 139, 162, 184, 204, 226, 244]
train_dataset, val_dataset, test_dataset = train_val_test_split(dataset, val_indices, test_indices)

batch_size = 32
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

print("Train dataset size:", len(train_dataset))
print("Validation dataset size:", len(val_dataset))
print("Test dataset size:", len(test_dataset))

Train dataset size: 210
Validation dataset size: 26
Test dataset size: 26


/tmp/ipykernel_39266/3945022793.py:26: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  data = torch.load("data_to_train/tensors.pt")


In [18]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from torchmetrics import Accuracy, F1Score


# Set CUDA_LAUNCH_BLOCKING for better error messages
os.environ['CUDA_LAUNCH_BLOCKING'] = '1'

class PixelClassifierCNN(nn.Module):
    def __init__(self, num_features=6, num_classes=18):
        super(PixelClassifierCNN, self).__init__()
        self.conv1 = nn.Conv2d(num_features, 32, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.conv3 = nn.Conv2d(64, 128, kernel_size=3, padding=1)
        self.fc = nn.Conv2d(128, num_classes, kernel_size=1)  # 1x1 Conv for per-pixel classification
    
    def forward(self, x):
        x = F.relu(self.conv1(x))
        x = F.relu(self.conv2(x))
        x = F.relu(self.conv3(x))
        x = self.fc(x)
        return x  # Output shape: (batch, num_classes, 26, 25)

# Dane wejściowe
batch_size = 32
num_classes = 18

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = PixelClassifierCNN(num_features=6, num_classes=num_classes)


batch_size = 32
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

model = model.to(device)
criterion = nn.CrossEntropyLoss(ignore_index=-100)
optimizer = optim.Adam(model.parameters(), lr=0.001)

acc = Accuracy(task='multiclass', num_classes=num_classes, ignore_index=-100).to(device)
f1 = F1Score(task='multiclass', num_classes=num_classes, ignore_index=-100).to(device)


# Trening
num_epochs = 100
for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    for inputs, labels in train_loader:
        inputs = inputs.float()
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(inputs)
        outputs = outputs.float().reshape(-1, num_classes)
        labels = labels.long().view(-1)
        try:
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            running_loss += loss.item()
        except RuntimeError as e:
            print(f"RuntimeError: {e}")
            print(f"Labels: {labels}")
            print(f"Outputs: {outputs}")
            raise e
    
    model.eval()
    val_loss, correct, total = 0.0, 0, 0
    all_preds, all_targets = [], []
    with torch.no_grad():
        for inputs, labels in val_loader:
            inputs = inputs.float()
            inputs, labels = inputs.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(inputs)
            outputs = outputs.float().reshape(-1, num_classes)
            labels = labels.long().view(-1)

            loss = criterion(outputs, labels)
            val_loss += loss.item()
            preds = torch.argmax(outputs, dim=1)
            valid_mask = labels != -100
            all_preds.append(preds[valid_mask])
            all_targets.append(labels[valid_mask])
    
    all_preds = torch.cat(all_preds)
    all_targets = torch.cat(all_targets)
    val_accuracy = acc(all_preds, all_targets).item()
    val_f1 = f1(all_preds, all_targets).item()
    
    print(f"Epoch {epoch+1}/{num_epochs}, Train Loss: {running_loss / len(train_loader):.4f}, Val Loss: {val_loss / len(val_loader):.4f}, Acc: {val_accuracy:.4f}, F1: {val_f1:.4f}")


Epoch 1/100, Train Loss: 2.8972, Val Loss: 2.8925, Acc: 0.0517, F1: 0.0517
Epoch 2/100, Train Loss: 2.8888, Val Loss: 2.8919, Acc: 0.0527, F1: 0.0527
Epoch 3/100, Train Loss: 2.8872, Val Loss: 2.8930, Acc: 0.0514, F1: 0.0514
Epoch 4/100, Train Loss: 2.8843, Val Loss: 2.8957, Acc: 0.0507, F1: 0.0507
Epoch 5/100, Train Loss: 2.8807, Val Loss: 2.9000, Acc: 0.0506, F1: 0.0506
Epoch 6/100, Train Loss: 2.8768, Val Loss: 2.9048, Acc: 0.0509, F1: 0.0509
Epoch 7/100, Train Loss: 2.8718, Val Loss: 2.9092, Acc: 0.0497, F1: 0.0497
Epoch 8/100, Train Loss: 2.8647, Val Loss: 2.9108, Acc: 0.0511, F1: 0.0511
Epoch 9/100, Train Loss: 2.8565, Val Loss: 2.9215, Acc: 0.0491, F1: 0.0491
Epoch 10/100, Train Loss: 2.8493, Val Loss: 2.9254, Acc: 0.0510, F1: 0.0510
Epoch 11/100, Train Loss: 2.8398, Val Loss: 2.9340, Acc: 0.0493, F1: 0.0493
Epoch 12/100, Train Loss: 2.8309, Val Loss: 2.9388, Acc: 0.0505, F1: 0.0505
Epoch 13/100, Train Loss: 2.8199, Val Loss: 2.9534, Acc: 0.0495, F1: 0.0495
Epoch 14/100, Train L